<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np

# =========================
# LOAD DATA
# =========================

def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        data = f.read()

    data = data.replace("\\n", "\n")
    return data


file_path = "hp1.txt"
text = load_data(file_path)

# =========================
# TOKENIZER
# =========================

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

tokenizer = Tokenizer(oov_token="<oov>")
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

tokens = tokenizer.texts_to_sequences([text])[0]

print("Total Tokens:", len(tokens))
print("Vocabulary Size:", total_words)

# =========================
# CREATE SEQUENCES
# =========================

seq_len = 50

input_sequences = []

for i in range(seq_len, len(tokens)):

    # 50 input + 1 target
    seq = tokens[i-seq_len:i+1]

    input_sequences.append(seq)

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# =========================
# POSITIONAL ENCODING
# =========================

class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, max_len, d_model):
        super().__init__()

        pos = np.arange(max_len)[:, np.newaxis]

        i = np.arange(d_model)[np.newaxis, :]

        angle_rates = 1 / np.power(
            10000,
            (2 * (i // 2)) / np.float32(d_model)
        )

        angle_rads = pos * angle_rates

        # even index
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

        # odd index
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.cast(
            angle_rads[np.newaxis, ...],
            dtype=tf.float32
        )

    def call(self, x):

        seq_len = tf.shape(x)[1]

        return x + self.pos_encoding[:, :seq_len, :]


# =========================
# DECODER BLOCK
# =========================

class DecoderBlock(tf.keras.layers.Layer):

    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])

        self.layernorm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.layernorm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)

        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):

        seq_len = tf.shape(x)[1]

        # causal mask
        mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)),
            -1,
            0
        )

        attn_output = self.mha(
            query=x,
            key=x,
            value=x,
            attention_mask=mask
        )

        attn_output = self.dropout1(
            attn_output,
            training=training
        )

        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        out2 = self.layernorm2(out1 + ffn_output)

        return out2


# =========================
# TRANSFORMER DECODER
# =========================

class TransformerDecoder(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_len,
        d_model=128,
        num_heads=4,
        dff=512,
        num_layers=2,
        dropout_rate=0.1
    ):

        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            max_len,
            d_model
        )

        self.decoder_layers = [
            DecoderBlock(
                d_model,
                num_heads,
                dff,
                dropout_rate
            )
            for _ in range(num_layers)
        ]

        self.dropout = tf.keras.layers.Dropout(dropout_rate)

        self.final_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x, training=False):

        # embedding
        x = self.embedding(x)

        # positional encoding
        x = self.pos_encoding(x)

        x = self.dropout(x, training=training)

        # decoder blocks
        for decoder in self.decoder_layers:
            x = decoder(x, training=training)

        # take last token only
        x = x[:, -1, :]

        # predict next word
        output = self.final_layer(x)

        return output


# =========================
# BUILD MODEL
# =========================

model = TransformerDecoder(
    vocab_size=total_words,
    max_len=seq_len,
    d_model=128,
    num_heads=4,
    dff=512,
    num_layers=2
)

# build model once
dummy = tf.random.uniform(
    (1, seq_len),
    maxval=100,
    dtype=tf.int32
)

model(dummy)

model.summary()

# =========================
# COMPILE
# =========================

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)

# =========================
# TRAIN
# =========================

history = model.fit(
    X,
    y,
    epochs=2,
    batch_size=32
)

# =========================
# GENERATE TEXT
# =========================

index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}


def generate_text(model, seed_text, next_words=30):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences(
            [seed_text]
        )[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=seq_len,
            padding='pre'
        )

        prediction = model.predict(
            token_list,
            verbose=0
        )

        predicted_id = np.argmax(prediction)

        output_word = index_to_word.get(
            predicted_id,
            ""
        )

        seed_text += " " + output_word

    return seed_text


# =========================
# TEST
# =========================

seed_text = "harry looked at"

generated_text = generate_text(
    model,
    seed_text,
    next_words=50
)

print("\nGenerated Text:\n")
print(generated_text)

Total Tokens: 11296
Vocabulary Size: 1795
X Shape: (11246, 50)
y Shape: (11246,)


Model: "transformer_decoder_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (1, 50, 128)           │       229,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_4           │ ?                      │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_block_8 (DecoderBlock)  │ ?                      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_block_9 (DecoderBlock)  │ ?                      │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (1, 1795)              │       231,555 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,253,379 (4.78 MB)

 Trainable params: 1,253,379 (4.78 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 118s 325ms/step - accuracy: 0.0327 - loss: 6.3949
Epoch 2/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 114s 323ms/step - accuracy: 0.0356 - loss: 6.1647

Generated Text:

harry looked at harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry harry
